In [2]:
import os

In [3]:
os.chdir("../")

In [4]:
%pwd

'/Users/shikhatiwari/Desktop/Medical-Chatbot-using-LLMs-LangChain-Pinecone-Flask-AWS'

In [23]:
## STEP 1: Extract Document of the source: in this prject we will download the pdf book using langchain

In [5]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [14]:
# Extract Text from pdf files
def load_pdf_files(data):
    loader = DirectoryLoader(
        data,
        glob="*.pdf",
        loader_cls=PyPDFLoader)
    
    documents = loader.load()
    return documents

In [15]:
extracted_data = load_pdf_files("/Users/shikhatiwari/Desktop/Medical-Chatbot-using-LLMs-LangChain-Pinecone-Flask-AWS/data")

In [17]:
len(extracted_data)

637

In [18]:
from typing import List
from langchain.schema import Document

def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
    """
    Given a list of document objects, return a new list of Document objects
    containing only 'source' in metadata and the original page_content.
    """

    minimal_docs: List[Document] =[]
    for doc in docs:
        src = doc.metadata.get("source")
        minimal_docs.append(
            Document(
                page_content=doc.page_content,
                metadata ={"source": src}
            )
        )

    return minimal_docs


In [19]:
minimal_docs = filter_to_minimal_docs(extracted_data)

In [22]:
minimal_docs[:5]

[Document(metadata={'source': '/Users/shikhatiwari/Desktop/Medical-Chatbot-using-LLMs-LangChain-Pinecone-Flask-AWS/data/Medical_book.pdf'}, page_content=''),
 Document(metadata={'source': '/Users/shikhatiwari/Desktop/Medical-Chatbot-using-LLMs-LangChain-Pinecone-Flask-AWS/data/Medical_book.pdf'}, page_content='The GALE\nENCYCLOPEDIA\nof MEDICINE\nSECOND EDITION'),
 Document(metadata={'source': '/Users/shikhatiwari/Desktop/Medical-Chatbot-using-LLMs-LangChain-Pinecone-Flask-AWS/data/Medical_book.pdf'}, page_content='The GALE\nENCYCLOPEDIA\nof MEDICINE\nSECOND EDITION\nJACQUELINE L. LONGE, EDITOR\nDEIRDRE S. BLANCHFIELD, ASSOCIATE EDITOR\nVOLUME\nA-B\n1'),
 Document(metadata={'source': '/Users/shikhatiwari/Desktop/Medical-Chatbot-using-LLMs-LangChain-Pinecone-Flask-AWS/data/Medical_book.pdf'}, page_content='STAFF\nJacqueline L. Longe, Project Editor\nDeirdre S. Blanchfield, Associate Editor\nChristine B. Jeryan, Managing Editor\nDonna Olendorf, Senior Editor\nStacey Blachford, Associate 

In [ ]:
## STEP 2: We have medical document and have also filtered the data now we will perform Chunking operation now

In [25]:
# Split the documents into smaller chunks
def text_split(minimal_docs):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size =500,
        chunk_overlap =20,
        #length_function=len
    )

    texts = text_splitter.split_documents(minimal_docs)
    return texts

In [26]:
texts_chunk = text_split(minimal_docs)

In [28]:
len(texts_chunk)

5859

In [29]:
texts_chunk[:5]

[Document(metadata={'source': '/Users/shikhatiwari/Desktop/Medical-Chatbot-using-LLMs-LangChain-Pinecone-Flask-AWS/data/Medical_book.pdf'}, page_content='The GALE\nENCYCLOPEDIA\nof MEDICINE\nSECOND EDITION'),
 Document(metadata={'source': '/Users/shikhatiwari/Desktop/Medical-Chatbot-using-LLMs-LangChain-Pinecone-Flask-AWS/data/Medical_book.pdf'}, page_content='The GALE\nENCYCLOPEDIA\nof MEDICINE\nSECOND EDITION\nJACQUELINE L. LONGE, EDITOR\nDEIRDRE S. BLANCHFIELD, ASSOCIATE EDITOR\nVOLUME\nA-B\n1'),
 Document(metadata={'source': '/Users/shikhatiwari/Desktop/Medical-Chatbot-using-LLMs-LangChain-Pinecone-Flask-AWS/data/Medical_book.pdf'}, page_content='STAFF\nJacqueline L. Longe, Project Editor\nDeirdre S. Blanchfield, Associate Editor\nChristine B. Jeryan, Managing Editor\nDonna Olendorf, Senior Editor\nStacey Blachford, Associate Editor\nKate Kretschmann, Melissa C. McDade, Ryan\nThomason, Assistant Editors\nMark Springer, Technical Specialist\nAndrea Lopeman, Programmer/Analyst\nBarba

In [30]:
## STEP 3: After chunking we will call embedding model to perform embedding

In [31]:
from langchain.embeddings import HuggingFaceEmbeddings

def download_embeddings():
    """ 
    Download and return the huggingface embedding model. 
    """
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    embeddings = HuggingFaceEmbeddings(
        model_name = model_name
    )

    return embeddings

embeddings = download_embeddings()

/var/folders/zx/1j6jp5hj0yq03nnpgzp5msn80000gn/T/ipykernel_14306/2276866430.py:8: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


In [32]:
embeddings

HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

In [34]:
vector = embeddings.embed_query("hello this is test for embeddings model")
print(vector)

[-0.011686321347951889, -0.048124637454748154, -0.010505739599466324, 0.01732753962278366, 0.07263346761465073, 0.04805712774395943, -0.05233575776219368, -0.01982734724879265, 0.012624974362552166, -0.047949548810720444, 0.06516627222299576, -0.013596123084425926, 0.06259375810623169, 0.053160663694143295, -0.11345572769641876, 0.011805242858827114, 0.07416199147701263, 0.01343910675495863, -0.05245864391326904, 0.0057475874200463295, -0.02050485834479332, 0.0010755164548754692, 0.013821078464388847, -0.051798805594444275, -0.005295619834214449, -0.030572587624192238, -0.02277933806180954, 0.04024980217218399, 0.0865168422460556, -0.061737269163131714, 0.09284989535808563, -0.024303453043103218, -0.02678891457617283, 0.11647556722164154, 0.03413161635398865, 0.010869906283915043, 0.015712566673755646, -0.042547423392534256, -0.029393620789051056, 0.017752015963196754, 0.013099477626383305, -0.034311920404434204, 0.015164715237915516, 0.0021264797542244196, 0.05655280873179436, -0.0247

In [ ]:
print(len(vector)) # which will be same as word_embedding_dimension

384


In [36]:
# Step 4: Load API keys

In [37]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [42]:
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

In [48]:
#Step 5: We will intialize pinecone client using the api key , 
# once we have the client use that to intialize the index and create the index by providing name, dimension, 
# metrics for similarity search and specification


In [43]:
from pinecone import Pinecone
pinecone_api_key = PINECONE_API_KEY

pc = Pinecone(api_key=pinecone_api_key)

In [44]:
pc

In [47]:
from pinecone import ServerlessSpec

index_name = "medical-chatbot"
if not pc.has_index(index_name):
    pc.create_index(
        name = index_name,
        dimension=384, #Dimension of the embedding model
        metric ="cosine", # Cosine similarity
        spec = ServerlessSpec(cloud="aws", region="us-east-1")
    )

index = pc.Index(index_name)

In [49]:
# Now we have successfully intialised the index in pinecone database

In [52]:
# Step 6: Add record in the pinecone database and then load the index

In [51]:
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents= texts_chunk,
    embedding=embeddings,
    index_name= index_name
)

In [53]:
# Load existing index
from langchain_pinecone import PineconeVectorStore

# Embed each chunk and upsert the embeddings into the pinecone index
docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embeddings
)

In [54]:
docsearch

In [ ]:
retriever = docsearch.as_retriever(search_type ="similarity", search_kwargs={"k":3})

In [56]:
retrieved_docs = retriever.invoke("what is acne")

In [62]:
retrieved_docs

[Document(id='f89988ed-50b0-463a-90f8-1ba36d5d352e', metadata={'source': '/Users/shikhatiwari/Desktop/Medical-Chatbot-using-LLMs-LangChain-Pinecone-Flask-AWS/data/Medical_book.pdf'}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 226\nAcne\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26'),
 Document(id='717d20df-9cb9-45ca-be89-f23c54bacd5f', metadata={'source': '/Users/shikhatiwari/Desktop/Medical-Chatbot-using-LLMs-LangChain-Pinecone-Flask-AWS/data/Medical_book.pdf'}, page_content='Acidosis see Respiratory acidosis; Renal\ntubular acidosis; Metabolic acidosis\nAcne\nDefinition\nAcne is a common skin disease characterized by\npimples on the face, chest, and back. It occurs when the\npores of the skin become clogged with oil, dead skin\ncells, and bacteria.\nDescription\nAcne vulgaris, the medical term for common acne, is\nthe most common skin disease. It affects nearly 17 million\npeople in the United States. While acne can arise at any'),
 Document(id='316ca678-68f6-47dd-a8a5-15daac86

In [61]:
from langchain_openai import ChatOpenAI

chatModel = ChatOpenAI(model ="gpt-4o")

In [64]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

In [65]:
system_prompt = (
    "You are an Medical assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)


In [66]:
question_answer_chain = create_stuff_documents_chain(chatModel, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [68]:
response = rag_chain.invoke({"input": "what is Acromegaly and gigantism?"})
print(response["answer"])

Acromegaly is a disorder caused by the abnormal release of growth hormone from the pituitary gland after bone growth has stopped, leading to increased growth in bones, soft tissue, and various disturbances throughout the body. Gigantism occurs when this growth hormone excess happens before the normal bone growth is complete, resulting in unusual height. Both conditions are relatively rare and can affect both men and women.


In [69]:
response = rag_chain.invoke({"input": "what is the Treatment of Acne?"})
print(response["answer"])

The treatment of acne depends on its severity. For mild noninflammatory acne, treatment includes reducing the formation of new comedones with topical agents like tretinoin, benzoyl peroxide, adapalene, or salicylic acid. In cases complicated by inflammation, topical antibiotics may be added, and improvement is typically seen in two to four weeks.
